In [ ]:
import numpy as np
import pandas as pd
import re, string tqdm
from collections import defaultdict
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import GaussianNB
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split

# Load preprocessed data
print("Loading preprocessed data...")
data = pd.read_csv('../DATASETS/SA_preprocessed.csv')

print(f"Dataset shape: {data.shape}")
print(f"Target distribution:\n{data['label'].value_counts()}")

# Prepare data
X = data['processed_text'].fillna('')
y = data['label'].values

print(f"\nTotal samples: {len(X)}")
print(f"Spam: {sum(y)}, Ham: {len(y) - sum(y)}")

Loading preprocessed data...
Dataset shape: (6045, 4)
Target distribution:
label
0    4149
1    1896
Name: count, dtype: int64

Total samples: 6045
Spam: 1896, Ham: 4149


In [ ]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"\nTraining samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")


Training samples: 4836
Testing samples: 1209


In [ ]:
print(X_train)

5643    cost effective mlm bizop phone screened leads ...
2113    bush orders sharon to obey un resolutions so u...
2918    re ilug mirror a website you could try httrack...
4146    neat net tricks standard issue number december...
2597    ilug optimising for pentiummmx i have been pla...
                              ...                        
3984    save an extra moneylimit number of compaq s po...
2553    ilug social re ilug linux the film moved from ...
4818    press release the business opportunity allianc...
2585    ilug samba login question i have a situation w...
5426    adv direct email blaster email addresses extra...
Name: processed_text, Length: 4836, dtype: object


# WORD2VEC FROM GENSIM LIBRARY

In [ ]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 30.5 MB/s eta 0:00:00


In [ ]:
from gensim.models import Word2Vec
import numpy as np

def get_gensim_sentence_vector(model, text):
    words = text.split()
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(model.vector_size)

def transform_texts(model, texts):
    return np.array([get_gensim_sentence_vector(model, t) for t in texts])

In [ ]:
# Train Gensim Word2Vec
print("Training Gensim Word2Vec...")
train_sentences = [str(t).lower().split() for t in X_train]
test_sentences = [str(t).lower().split() for t in X_test]
full_corpus = train_sentences + test_sentences

gensim_w2v = Word2Vec(sentences=full_corpus,
                      vector_size=100,
                      window=5,
                      min_count=2,
                      workers=4,
                      epochs=5) # By default uses CBOW for training

# Transform data
print("Transforming data...")
X_train_w2v = transform_texts(gensim_w2v, X_train)
X_test_w2v  = transform_texts(gensim_w2v, X_test)

print(f"Word2Vec training data shape: {X_train_w2v.shape}")
print(f"Word2Vec test data shape: {X_test_w2v.shape}")

Training Gensim Word2Vec...
Transforming data...
Word2Vec training data shape: (4836, 100)
Word2Vec test data shape: (1209, 100)


In [ ]:
print("\nWord2Vec + Naive Bayes")
w2v_nb = GaussianNB()
w2v_nb.fit(X_train_w2v, y_train)
w2v_nb_pred = w2v_nb.predict(X_test_w2v)

print(f"Accuracy: {accuracy_score(y_test, w2v_nb_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, w2v_nb_pred):.4f}")


Word2Vec + Naive Bayes
Accuracy: 0.8710
F1 Score: 0.7942


In [ ]:
print("Word2Vec + SVM")
w2v_svm = SVC(kernel='rbf', C=1.0, random_state=42)
w2v_svm.fit(X_train_w2v, y_train)
w2v_svm_pred = w2v_svm.predict(X_test_w2v)

print(f"Accuracy: {accuracy_score(y_test, w2v_svm_pred):.4f}")
print(f"F1 Score: {f1_score(y_test, w2v_svm_pred):.4f}")

Word2Vec + SVM
Accuracy: 0.9702
F1 Score: 0.9525


# IMPLEMENTATION FROM SCRATCH

Code partially borrowed from: https://www.tensorflow.org/text/tutorials/word2vec

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers

# Configs
SEED          = 42
VOCAB_SIZE    = 30000
SEQ_LEN       = 10
EMBEDDING_DIM = 100
WINDOW_SIZE   = 2
NUM_NS        = 4
BATCH_SIZE    = 1024
BUFFER_SIZE   = 10_000
EPOCHS        = 1000
AUTOTUNE      = tf.data.AUTOTUNE

# Load Data
print("Loading data...")
data = pd.read_csv('/content/SA_optimized_preprocessed_final.csv')
X    = data['processed_text'].fillna('')
y    = data['label'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=SEED, stratify=y
)

# Build TF Text Dataset
corpus  = X_train.tolist() + X_test.tolist()
text_ds = tf.data.Dataset.from_tensor_slices(corpus).filter(
    lambda x: tf.cast(tf.strings.length(x), bool)
)

# Vectorisation Layer
def custom_standardize(input_data):
    return tf.strings.regex_replace(
        tf.strings.lower(input_data),
        '[%s]' % re.escape(string.punctuation), ''
    )

vectorize_layer = layers.TextVectorization(
    standardize=custom_standardize,
    max_tokens=VOCAB_SIZE,
    output_mode='int',
    output_sequence_length=SEQ_LEN,
)
vectorize_layer.adapt(text_ds.batch(1024))
vocab = vectorize_layer.get_vocabulary()
word_to_idx = {w: i for i, w in enumerate(vocab)}

# Build Integer Sequences
text_vector_ds = text_ds.batch(1024).prefetch(AUTOTUNE).map(vectorize_layer).unbatch()
sequences = list(text_vector_ds.as_numpy_iterator())

# Skip-Gram + Negative Sampling
def generate_training_data(sequences, window_size, num_ns, vocab_size, seed):
    targets, contexts, labels = [], [], []
    vocab_size = int(vocab_size)

    for seq in tqdm.tqdm(sequences, desc="Generating skip-grams"):
        pos_pairs, _ = tf.keras.preprocessing.sequence.skipgrams(
            seq,
            vocabulary_size=vocab_size,
            sampling_table=None,
            window_size=window_size,
            negative_samples=0,
            seed=int(seed)
        )
        for target_word, context_word in pos_pairs:
            context_class = tf.expand_dims(tf.constant([context_word], dtype="int64"), 1)
            neg_samples, _, _ = tf.random.log_uniform_candidate_sampler(
                true_classes=context_class, num_true=1,
                num_sampled=num_ns, unique=True,
                range_max=vocab_size, seed=seed,
                name="negative_sampling",
            )
            context = tf.concat([tf.squeeze(context_class, 1), neg_samples], 0)
            label   = tf.constant([1] + [0] * num_ns, dtype="int64")
            targets.append(target_word)
            contexts.append(context)
            labels.append(label)

    return np.array(targets), np.array(contexts), np.array(labels)

targets, contexts, labels = generate_training_data(
    sequences=sequences,
    window_size=WINDOW_SIZE,
    num_ns=NUM_NS,
    vocab_size=VOCAB_SIZE,
    seed=SEED,
)

# tf.data Pipeline
dataset = tf.data.Dataset.from_tensor_slices(((targets, contexts), labels))
dataset = dataset.shuffle(BUFFER_SIZE).batch(BATCH_SIZE, drop_remainder=True).cache().prefetch(AUTOTUNE)

# Subclassed Word2Vec Model
class Word2Vec(tf.keras.Model):
    def __init__(self, vocab_size, embedding_dim):
        super().__init__()
        self.target_embedding  = layers.Embedding(vocab_size, embedding_dim, name="w2v_embedding")
        self.context_embedding = layers.Embedding(vocab_size, embedding_dim)

    def call(self, pair):
        target, context = pair
        if len(target.shape) == 2:
            target = tf.squeeze(target, axis=1)
        word_emb    = self.target_embedding(target)
        context_emb = self.context_embedding(context)
        dots        = tf.einsum('be,bce->bc', word_emb, context_emb)
        return dots

word2vec = Word2Vec(VOCAB_SIZE, EMBEDDING_DIM)
word2vec.compile(optimizer='adam', loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True))
word2vec.fit(dataset, epochs=EPOCHS)

# 8. Extract Embedding Matrix
embedding_matrix = word2vec.get_layer('w2v_embedding').get_weights()[0]

# 9. Sentence Vectoriser
def sentence_to_vector(text):
    if not isinstance(text, str): return np.zeros(EMBEDDING_DIM)
    tokens = re.sub('[%s]' % re.escape(string.punctuation), '', text.lower()).split()
    vecs = [embedding_matrix[word_to_idx[t]] for t in tokens if t in word_to_idx and word_to_idx[t] > 1]
    return np.mean(vecs, axis=0) if vecs else np.zeros(EMBEDDING_DIM)

def transform(texts):
    return np.array([sentence_to_vector(t) for t in texts])

X_train_w2v = transform(X_train.tolist())
X_test_w2v  = transform(X_test.tolist())

# 10. Classifiers
def evaluate(name, model):
    model.fit(X_train_w2v, y_train)
    pred = model.predict(X_test_w2v)
    print(f"\n{name}")
    print(f"Accuracy : {accuracy_score(y_test, pred):.4f}")
    print(f"f1 : {f1_score(y_test, pred):.4f}")
    print(classification_report(y_test, pred, target_names=['Ham', 'Spam']))

evaluate("Word2Vec + Naive Bayes", GaussianNB())
evaluate("Word2Vec + SVM (RBF)", SVC(kernel='rbf', C=1.0, random_state=SEED))

Loading data...


Generating skip-grams: 100%|██████████| 6043/6043 [03:41<00:00, 27.29it/s]


Epoch 1/3
200/200 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - loss: 1.4591
Epoch 2/3
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.1753
Epoch 3/3
200/200 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - loss: 1.0074

── Word2Vec + Naive Bayes ──────────────────────────
Accuracy : 0.6964
              precision    recall  f1-score   support

         Ham       0.73      0.88      0.80       830
        Spam       0.53      0.29      0.37       379

    accuracy                           0.70      1209
   macro avg       0.63      0.59      0.59      1209
weighted avg       0.67      0.70      0.67      1209


── Word2Vec + SVM (RBF) ──────────────────────────
Accuracy : 0.9313
              precision    recall  f1-score   support

         Ham       0.95      0.95      0.95       830
        Spam       0.90      0.88      0.89       379

    accuracy                           0.93      1209
   macro avg       0.92      0.92      0.92      1209
weighted avg       0.93      0.93      0.93      1209


── Wo